In [2]:
POKEMON_SPECIES = {
    "pikachu": "Pikachu",
    "charizard": "Charizard",
    "bulbasaur": "Bulbasaur",
    "mewtwo": "Mewtwo"
}


In [9]:
ATTACK_KEYWORDS = [
    "neutralize",
    "neutralized",
    "eliminate",
    "destroy",
    "attack",
    "take out",
    "remove",
    "kill",
    "terminate"
]


In [3]:
PROTECT_KEYWORDS = [
    "must not",
    "do not",
    "avoid",
    "protect"
    "protected",
    "not be harmed",
    "should not",
    "cannot be harmed"
]


In [4]:
import re

def normalize_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\.]", "", text)
    return text


In [5]:
def split_sentences(text: str):
    return [s.strip() for s in text.split(".") if s.strip()]


In [8]:
def parse_mission(text: str):
    text = normalize_text(text)

    targets = set()
    protected = set()

    sentences = split_sentences(text)

    for sentence in sentences:
        # detect intent in this sentence
        has_attack_intent = any(word in sentence for word in ATTACK_KEYWORDS)
        has_protect_intent = any(word in sentence for word in PROTECT_KEYWORDS)

        for key, pokemon in POKEMON_SPECIES.items():
            # handles singular & plural automatically (substring match)
            if key in sentence:
                if has_attack_intent:
                    targets.add(pokemon)
                if has_protect_intent:
                    protected.add(pokemon)

    # SAFETY RULE: protected always overrides target
    targets -= protected

    return {
        "targets": sorted(list(targets)),
        "protected": sorted(list(protected))
    }


In [10]:
mission_text = """
HQ has detected suspicious Bulbasaur activity in the region.
All Bulbasaurs are to be neutralized immediately.
Pikachu and Charizard are also present in the area and must not be harmed.
"""

result = parse_mission(mission_text)
print(result)


{'targets': ['Bulbasaur'], 'protected': ['Charizard', 'Pikachu']}


In [11]:
TEST_CASES = [
    {
        "id": 1,
        "text": "All Bulbasaurs are to be neutralized immediately.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": []
        }
    },
    {
        "id": 2,
        "text": "Pikachu and Mewtwo must be eliminated before they escape.",
        "expected": {
            "targets": ["Pikachu", "Mewtwo"],
            "protected": []
        }
    },
    {
        "id": 3,
        "text": "Charizard and Pikachu are protected assets and must not be harmed.",
        "expected": {
            "targets": [],
            "protected": ["Charizard", "Pikachu"]
        }
    },
    {
        "id": 4,
        "text": "All Bulbasaurs are to be destroyed. Pikachu must not be harmed.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 5,
        "text": "Mewtwo has been spotted in the area. Charizard should not be attacked. Eliminate all Pikachu immediately.",
        "expected": {
            "targets": ["Pikachu"],
            "protected": ["Charizard"]
        }
    },
    {
        "id": 6,
        "text": (
            "Recon reports several Pokémon in the region. "
            "Bulbasaur activity is escalating and must be neutralized. "
            "Pikachu and Charizard are friendly units and should not be harmed under any circumstances."
        ),
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu", "Charizard"]
        }
    },
    {
        "id": 7,
        "text": "Mewtwo has been seen flying overhead. No further action required.",
        "expected": {
            "targets": [],
            "protected": []
        }
    },
    {
        "id": 8,
        "text": "Bulbasaur must be eliminated. Bulbasaur is also protected due to diplomatic reasons.",
        "expected": {
            "targets": [],
            "protected": ["Bulbasaur"]
        }
    },
    {
        "id": 9,
        "text": "Do not attack Pikachu. Bulbasaurs are hostile and must be removed.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 10,
        "text": "Eliminate Pikachu and Bulbasaur but avoid Charizard at all costs.",
        "expected": {
            "targets": ["Pikachu", "Bulbasaur"],
            "protected": ["Charizard"]
        }
    },
    {
        "id": 11,
        "text": "HQ authorizes removal of all Bulbasaur units. Pikachu presence is confirmed but they cannot be harmed.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 12,
        "text": "Pikachus and Bulbasaurs are present. Bulbasaurs must be neutralized. Pikachus should not be attacked.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 13,
        "text": "Pikachu, Charizard, Bulbasaur, and Mewtwo are all protected units.",
        "expected": {
            "targets": [],
            "protected": ["Bulbasaur", "Charizard", "Mewtwo", "Pikachu"]
        }
    },
    {
        "id": 14,
        "text": "Eliminate Pikachu. Destroy Charizard. Neutralize Bulbasaur. Take out Mewtwo.",
        "expected": {
            "targets": ["Bulbasaur", "Charizard", "Mewtwo", "Pikachu"],
            "protected": []
        }
    },
    {
        "id": 15,
        "text": "Bulbasaur must be neutralized, destroyed, and removed immediately.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": []
        }
    },
    {
        "id": 16,
        "text": "Pikachu should not, under any circumstances, be attacked. Bulbasaur however must be attacked.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 17,
        "text": "Charizard must not be harmed, but eliminate all Bulbasaurs.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": ["Charizard"]
        }
    },
    {
        "id": 18,
        "text": "Weather conditions are poor. Visibility is low. Pikachu should not be harmed. Proceed to eliminate Mewtwo.",
        "expected": {
            "targets": ["Mewtwo"],
            "protected": ["Pikachu"]
        }
    },
    {
        "id": 19,
        "text": "It is recommended that Bulbasaur units be taken out. Pikachu units are friendly.",
        "expected": {
            "targets": ["Bulbasaur"],
            "protected": []
        }
    },
    {
        "id": 20,
        "text": "Stand by. Await further instructions.",
        "expected": {
            "targets": [],
            "protected": []
        }
    }
]


In [12]:
def run_nlp_tests(test_cases):
    passed = 0

    for case in test_cases:
        result = parse_mission(case["text"])
        expected = case["expected"]

        success = (
            set(result["targets"]) == set(expected["targets"]) and
            set(result["protected"]) == set(expected["protected"])
        )

        status = "✅ PASS" if success else "❌ FAIL"

        print(f"Test Case {case['id']} — {status}")
        print("Text:")
        print(case["text"])
        print("Expected:", expected)
        print("Got:     ", result)
        print("-" * 60)

        if success:
            passed += 1

    print(f"\nSummary: {passed}/{len(test_cases)} tests passed.")


In [11]:
run_nlp_tests(TEST_CASES)

Test Case 1 — ✅ PASS
Text:
All Bulbasaurs are to be neutralized immediately.
Expected: {'targets': ['Bulbasaur'], 'protected': []}
Got:      {'targets': ['Bulbasaur'], 'protected': []}
------------------------------------------------------------
Test Case 2 — ✅ PASS
Text:
Pikachu and Mewtwo must be eliminated before they escape.
Expected: {'targets': ['Pikachu', 'Mewtwo'], 'protected': []}
Got:      {'targets': ['Mewtwo', 'Pikachu'], 'protected': []}
------------------------------------------------------------
Test Case 3 — ✅ PASS
Text:
Charizard and Pikachu are protected assets and must not be harmed.
Expected: {'targets': [], 'protected': ['Charizard', 'Pikachu']}
Got:      {'targets': [], 'protected': ['Charizard', 'Pikachu']}
------------------------------------------------------------
Test Case 4 — ✅ PASS
Text:
All Bulbasaurs are to be destroyed. Pikachu must not be harmed.
Expected: {'targets': ['Bulbasaur'], 'protected': ['Pikachu']}
Got:      {'targets': ['Bulbasaur'], 'protect